In [ ]:
# 01 · Exploración del login · saucedemo (PROD)
#
# Casos negativos del login, login exitoso con standard_user y logout.
#
# Ejecuta la celda. Debajo aparece el log con fecha (dd-mm-aaaa), hora y ambiente.
# El log se guarda en reports/logs/<AMBIENTE>/<aaaa>/<mm>/<dd>/ y las capturas en reports/evidencias/.
# Chrome se abre en modo visible (para ocultarlo: variable de entorno HEADLESS=1).

import contextlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # permite importar pages/ y utils/

from pages import InventoryPage, LoginPage
from utils.config import AMBIENTE, BASE_URL, PASSWORD, USERS
from utils.driver_factory import create_driver
from utils.evidence import save_screenshot
from utils.logger import get_logger, setup_execution_log

log_file = setup_execution_log("01_exploracion_login")
log = get_logger()
log.info("===== INICIO | Exploración del login =====")
log.info("Ambiente: %s | Sitio: %s", AMBIENTE, BASE_URL)

driver = create_driver()
try:
    login_page = LoginPage(driver)

    # 1. Casos negativos: (descripción, usuario, password, error esperado)
    casos_negativos = [
        ("Usuario bloqueado", USERS["locked_out"], PASSWORD, "Sorry, this user has been locked out."),
        ("Password incorrecta", USERS["standard"], "clave_incorrecta", "Username and password do not match"),
        ("Sin usuario", "", PASSWORD, "Username is required"),
        ("Sin password", USERS["standard"], "", "Password is required"),
    ]
    fallos = 0
    for descripcion, usuario, clave, esperado in casos_negativos:
        login_page.load().login(usuario, clave)
        mensaje = login_page.error_message()
        if esperado in mensaje:
            log.info("[OK] %s -> %s", descripcion, mensaje)
        else:
            fallos += 1
            log.error("[FALLO] %s -> se esperaba '%s' y se obtuvo '%s'", descripcion, esperado, mensaje)
    save_screenshot(driver, "01_casos_negativos")

    # 2. Login exitoso
    inventory = login_page.load().login_as(USERS["standard"])
    assert inventory.title() == "Products", f"Título inesperado: {inventory.title()}"
    log.info("[OK] Login exitoso -> página '%s' (%s)", inventory.title(), inventory.current_url)
    save_screenshot(driver, "01_login_exitoso")

    # 3. Logout
    login_page = inventory.logout()
    assert login_page.is_displayed(), "No se volvió a la pantalla de login"
    log.info("[OK] Logout -> se volvió a la pantalla de login")

    assert fallos == 0, f"{fallos} caso(s) negativo(s) fallaron"
    log.info("RESULTADO: todas las validaciones OK")
except Exception:
    log.exception("RESULTADO: la ejecución falló")
    with contextlib.suppress(Exception):
        save_screenshot(driver, "01_error")
    raise
finally:
    driver.quit()
    log.info("===== FIN | Log guardado en %s =====", log_file)